# Code 10.2.1: The gridworld MDP

The grid, its walls, goal, and pit, and a class that precomputes the deterministic transition table `next_state[s, a]` and the reward table `reward[s, a]`. Only non-wall cells are states, numbered 0 to 21.


In [ ]:
import numpy as np

ROWS, COLS = 5, 5
WALLS = {(1, 1), (1, 2), (3, 3)}
GOAL, PIT = (0, 4), (2, 3)
START = (4, 0)
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]          # up, down, left, right

class GridWorld:
    def __init__(self):
        self.cells = [(r, c) for r in range(ROWS) for c in range(COLS)
                      if (r, c) not in WALLS]
        self.index = {cell: i for i, cell in enumerate(self.cells)}
        self.n_states, self.n_actions = len(self.cells), len(ACTIONS)
        self.terminal = np.array([cell in (GOAL, PIT) for cell in self.cells])
        # Deterministic transitions: next_state[s, a] and reward[s, a].
        self.next_state = np.zeros((self.n_states, self.n_actions), int)
        self.reward = np.zeros((self.n_states, self.n_actions))
        for s, (r, c) in enumerate(self.cells):
            for a, (dr, dc) in enumerate(ACTIONS):
                nr, nc = r + dr, c + dc
                if not (0 <= nr < ROWS and 0 <= nc < COLS) or (nr, nc) in WALLS:
                    nr, nc = r, c                      # bump: stay put
                self.next_state[s, a] = self.index[(nr, nc)]
                self.reward[s, a] = {GOAL: 1.0, PIT: -1.0}.get((nr, nc), 0.0)

    def step(self, s, a):
        """Return (next_state, reward, done) for action a in state s."""
        s2 = self.next_state[s, a]
        return s2, self.reward[s, a], bool(self.terminal[s2])

    def to_grid(self, v, fill=np.nan):
        """Arrange a per-state vector as a ROWS x COLS array (walls = fill)."""
        g = np.full((ROWS, COLS), fill, dtype=float)
        for s, (r, c) in enumerate(self.cells):
            g[r, c] = v[s]
        return g

def random_policy(env):
    return np.full((env.n_states, env.n_actions), 1.0 / env.n_actions)
